# 1.0 – Thu thập và chuẩn hóa dữ liệu (TV2, tầng dữ liệu)

Từ 54 đề cương PDF scan đến bộ dữ liệu v1: bảng kê, chữ OCR sạch, bảng CLO thô, JSON đề cương (10 mục) và `sections_labeled.csv`.
Notebook chỉ **đọc** kết quả của các script trong `src/syllabus_core/data/` và kiểm tra lại; chạy lại từ đầu bằng *Restart & Run All*.

Thứ tự script: `make_inventory` → `pdf_to_images` → `ocr_run` (Tesseract + PaddleOCR) → `clean_text` → `extract_clos` → `split_sections` → `make_dataset` → `validate_json`.

In [1]:
import csv, json, statistics, sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists():  # notebook nằm ở notebooks/exploratory/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src/syllabus_core/data"))
from split_sections import SECTIONS
from validate_json import validate_all

def read_csv(p):
    with open(ROOT / p, encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

print("Thư mục gốc:", ROOT.name)

Thư mục gốc: syllabus-rag


## 1. Bảng kê đề cương (`inventory.csv`)

In [3]:
inv = read_csv("data/raw/inventory.csv")
dups = [r for r in inv if r["duplicate_of"]]
no_code = [r["file_name"] for r in inv if not r["course_code"]]
print(f"{len(inv)} file PDF; {len(dups)} bản trùng; {len(inv) - len(dups)} môn dùng được")
print("Trùng:", [(r["file_name"][:12], "->", r["duplicate_of"][:12]) for r in dups])
print("Thiếu mã môn:", no_code)
print("Tổng số trang:", sum(int(r["n_pages"]) for r in inv if r["n_pages"] and not r["duplicate_of"]))

57 file PDF; 3 bản trùng; 54 môn dùng được
Trùng: [('66 - 841479 ', '->', '63 - 841479 '), ('81 - 841434 ', '->', '62 - 841434 '), ('82 - 841407 ', '->', '48 - 841407 ')]
Thiếu mã môn: ['74 - Công nghệ. NET.pdf', '80 - Hệ thống thông tin doanh nghiệp.pdf']
Tổng số trang: 488


## 2. Bảng CLO thô (`clos_raw.csv`)

In [5]:
clos = read_csv("data/interim/clos_raw.csv")
per = Counter(r["course_code"] for r in clos)
print(f"{len(clos)} CLO, {len(per)} môn; số CLO mỗi môn: min {min(per.values())}, trung vị {statistics.median(per.values())}, max {max(per.values())}")
print("level_raw rỗng:", sum(not r["level_raw"] for r in clos), "| ocr_ok = False:", sum(r["ocr_ok"] != "True" for r in clos))
print("Các dạng level_raw:", sorted({r["level_raw"] for r in clos}))

411 CLO, 54 môn; số CLO mỗi môn: min 3, trung vị 6.0, max 33
level_raw rỗng: 12 | ocr_ok = False: 0
Các dạng level_raw: ['', '1', '2', '2,0', '2,3,4,5', '2,5', '2.5', '3', '3,0', '3,4', '3.0', '3.5', '4', '4,5', '4.0', '4.5', '5', '5.0']


## 3. JSON đề cương (`data/processed/syllabi/`)

Trường trích bằng luật từ mục 1 rồi **đối chiếu lại với ảnh trang 1** (sửa tay ghi ở `data/interim/syllabus_fixes.json`).

In [7]:
docs = {p.stem: json.loads(p.read_text(encoding="utf-8")) for p in sorted((ROOT / "data/processed/syllabi").glob("*.json"))}
print(len(docs), "file JSON")
print("Tín chỉ:", sorted(Counter(d["credits"] for d in docs.values()).items()))
print("Khối kiến thức:", dict(Counter(d["knowledge_block"] for d in docs.values())))
print("Môn có tiên quyết:", sum(bool(d["prerequisites"]) for d in docs.values()))
print("Môn đủ 10 mục:", sum(all(d["sections"][s] is not None for s in SECTIONS) for d in docs.values()))
print("Số CLO khớp clos_raw ở mọi môn:", all(len(d["clos"]) == per[k] for k, d in docs.items()))

54 file JSON
Tín chỉ: [(2, 3), (3, 22), (4, 28), (6, 1)]
Khối kiến thức: {'co_so_nganh': 7, 'nganh': 20, 'chuyen_nganh': 21, None: 3, 'dai_cuong': 3}
Môn có tiên quyết: 24
Môn đủ 10 mục: 54
Số CLO khớp clos_raw ở mọi môn: True


## 4. Kiểm theo schema

In [9]:
res = validate_all(ROOT / "data/processed/syllabi", ROOT / "docs/schema_syllabus.json")
bad = {k: v for k, v in res.items() if v}
print(f"{len(res) - len(bad)}/{len(res)} file hợp lệ")
for k, v in bad.items():
    print(k, v)
assert not bad

54/54 file hợp lệ


## 5. Nhãn mục (`sections_labeled.csv`)

`section_regex` do luật tiêu đề gán; `section_gold` là cột để người rà (đoạn ở ranh giới nằm ở `data/interim/sections_review.csv`).

In [11]:
sec = read_csv("data/processed/sections_labeled.csv")
print(len(sec), "đoạn,", len({r["course_code"] for r in sec}), "môn (yêu cầu ≥ 1.000 đoạn)")
for s, n in Counter(r["section_gold"] for r in sec).most_common():
    print(f"  {s:26}{n:5}")
changed = sum(r["section_gold"] != r["section_regex"] for r in sec)
print("Đoạn đã sửa tay (gold khác regex):", changed)
print("Đoạn cần rà ưu tiên:", len(read_csv("data/interim/sections_review.csv")))

5693 đoạn, 54 môn (yêu cầu ≥ 1.000 đoạn)
  teaching_schedule          1951
  course_content              952
  course_learning_outcomes    532
  assessment                  515
  responsible_unit            388
  course_objectives           367
  general_info                311
  learning_resources          304
  course_description          130
  other                       129
  course_policies             114
Đoạn đã sửa tay (gold khác regex): 0
Đoạn cần rà ưu tiên: 2851


## 6. Hạn chế đã biết

- Số tín chỉ, số tiết, khối kiến thức, tiên quyết: luật trên chữ OCR chỉ đúng một phần nên đã đối chiếu tay với ảnh; chưa có người thứ hai kiểm.
- `section_gold` mới là bản sao của `section_regex` ở những đoạn chưa rà; chỉ dùng làm nhãn chuẩn sau khi rà xong.
- Bảng lịch giảng (mục 7) bị tách thành nhiều đoạn ngắn vì mỗi ô bảng là một dòng OCR.